## Imports and Supabase Connection

In [2]:
import os
import pandas as pd
from supabase import create_client
from dotenv import load_dotenv
from pathlib import Path
import plotly.express as px
import statsmodels.api as sm
import statsmodels.formula.api as smf

%matplotlib inline
# Resolve the project root (one level above `/notebooks`).
project_root = Path.cwd().parent
env_path = project_root / ".env"

load_dotenv(dotenv_path=env_path, override=True)

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

# Optional sanity check.
print("Loaded key starts with:", SUPABASE_KEY[:5])


Loaded key starts with: eyJhb


## Incident

In [3]:
# Read the incident table in 1,000-row pages to avoid timeouts and memory spikes.
page_size = 1000
# Accumulate rows until Supabase returns an empty page.
all_rows = []
start = 0

while True:
    response = (
        supabase
        .table("incident")
        .select("*")
        .order("Incident_ID")   # Required for stable pagination.
        .range(start, start + page_size - 1)
        .execute()
    )

    data = response.data
    if not data:
        break

    all_rows.extend(data)
    start += page_size

incident_df = pd.DataFrame(all_rows)
incident_df["Number_Victims"] = pd.to_numeric(incident_df["Number_Victims"], errors="coerce")

# Quick validation of total row count.
print (len(incident_df))
print (incident_df.head(2))


3136
     Incident_ID  Month  Day  Year                       Date  \
0  19660311NCIRC      3   11  1966  1966-03-11T00:00:00+00:00   
1  19660314TXCAW      3   14  1966  1966-03-14T00:00:00+00:00   

                             School Victims_Killed Victims_Wounded  \
0  Irwing Avenue Junior High School              0               1   
1                Carver High School              1               0   

   Number_Victims Shooter_Killed  ... Preplanned SRO_School  \
0               1              0  ...         No              
1               1              0  ...         No        Yes   

  Security_Screening     Screening_Outcome Shots_Fired School_Lockdown  \
0                                                    7                   
1       Armed Guards  Outside/Off-Property           3                   

         LAT         LNG Campus_Type Zipcode  
0  35.237069  -80.850227               28202  
1   31.57954  -97.130303               76704  

[2 rows x 50 columns]


# Model Exploring

## Incidents

### Apply the best subset selection approach to incident data. We wish to predict risk based on a variety of features/predictors

In [4]:
valid_states = {
"AL","AK","AZ","AR","CA","CO","CT","DE","FL","GA",
"HI","ID","IL","IN","IA","KS","KY","LA","ME","MD",
"MA","MI","MN","MS","MO","MT","NE","NV","NH","NJ",
"NM","NY","NC","ND","OH","OK","OR","PA","RI","SC",
"SD","TN","TX","UT","VT","VA","WA","WV","WI","WY","DC"
}

invalid = incident_df[~incident_df["State"].isin(valid_states)]
invalid["State"].unique()

<StringArray>
[             'Winter',             'Houston',         'Rogersville',
           'Hahnville',             'Artesia',           'Pensacola',
           'St. Louis',              'Dallas',          'Fort Worth',
              'Spring',                'Fall',            'Hartford',
            'Anniston',               'Miami',              'Aldine',
            'Portland',        'New Rochelle',        'Indianapolis',
               'Tampa',             'Bristol',          'Fort Myers',
            'Winnetka',             'Detroit',                'Oahu',
         'Baton Rouge',         'Port Arthur',      'Virginia Beach',
            'Stockton',          'Washington',             'Slidell',
           'Rock Hill',              'Merced',            'Amarillo',
          'Montgomery',                   '4',              'Summer',
 'San Fernando Valley',          'Manchester',            'New York',
               'Omaha',                  'VI',                    '',
      

## Sample Restriction (1987–Present)

The panel is restricted to 1987 onward due to a structural break in the underlying data-generating process prior to 1987.

Pre-1987 observations are not comparable in measurement, reporting coverage, and policy codification. Including those years would violate panel homogeneity assumptions and introduce non-structural noise into fixed effects and dynamic specifications.

This restriction is not outcome-based filtering. It is a data consistency adjustment to ensure:

- Comparable incident classification
- Consistent enrollment coverage
- Reliable policy coding
- Stable variance structure

All models are therefore estimated on the 1987–present balanced policy-enrollment panel.


## Incidents from 1987

In [5]:
incident_df_from_1987 = incident_df[incident_df["State"].isin(valid_states)]
print(incident_df_from_1987.shape) # incident_df_from_1987.shape
print(incident_df_from_1987["State"].unique())

(3070, 50)
<StringArray>
['NC', 'TX', 'CA', 'NY', 'FL', 'PA', 'MN', 'TN', 'IL', 'VT', 'HI', 'OK', 'KY',
 'DC', 'WI', 'OH', 'AR', 'DE', 'UT', 'ID', 'IA', 'MI', 'NJ', 'AZ', 'VA', 'MD',
 'NM', 'LA', 'NV', 'IN', 'AL', 'MO', 'CT', 'SC', 'GA', 'CO', 'MA', 'WV', 'MS',
 'OR', 'MT', 'KS', 'WA', 'WY', 'RI', 'NH', 'NE', 'AK', 'ME', 'SD', 'ND']
Length: 51, dtype: str


In [6]:
# incident count per state-year combination
incident_counts = (
    incident_df_from_1987
    .groupby(["State", "Year"])
    .size()
    .reset_index(name="incident_count")
)

# Ensure Victims_Killed is numeric before aggregation
incident_df_from_1987["Victims_Killed"] = pd.to_numeric(
    incident_df_from_1987["Victims_Killed"],
    errors="coerce"
)

# Replace any non-numeric values with 0
incident_df_from_1987["Victims_Killed"] = (
    incident_df_from_1987["Victims_Killed"].fillna(0)
)

In [7]:
# supabase.table("state_year_enrollment").select("*").execute()

In [8]:
response = supabase.table("Enrollment Data Final").select("*").limit(1).execute(); print(response.data)

[{'school_id': 2100572, 'state': 'Kentucky', 'year': 1987, 'total_students': 272}]


In [9]:
all_rows = []
start = 0
batch_size = 1000

while True:
    response = (
        supabase
        .table("enrollment_state_year_mat")
        .select("*")
        .range(start, start + batch_size - 1)
        .execute()
    )
    
    data = response.data
    
    if not data:
        break
        
    all_rows.extend(data)
    start += batch_size

enrollment_df = pd.DataFrame(all_rows)

print(enrollment_df.shape)

(2548, 3)


In [10]:
# Keep only incident years that exist in enrollment data
incident_counts = incident_counts[
    incident_counts["Year"] >= 1987
]

In [11]:
enrollment_df


,state,year,total_students
0,RHODE ISLAND,2020,10006
1,Louisiana,2025,569440
2,Montana,1999,160294
3,Georgia,2015,1721666
4,Rhode Island,2016,139409
...,...,...,...
2543,District of Columbia,2013,75819
2544,Kentucky,2014,683957
2545,GEORGIA,2020,87127
2546,DISTRICT OF COLUMBIA,2015,1671


In [12]:
# Convert full state names to uppercase for consistent mapping
enrollment_df["state"] = enrollment_df["state"].str.upper()


# Map full state names to two-letter abbreviations
state_abbrev = {
    'ALABAMA': 'AL', 'ALASKA': 'AK', 'ARIZONA': 'AZ', 'ARKANSAS': 'AR',
    'CALIFORNIA': 'CA', 'COLORADO': 'CO', 'CONNECTICUT': 'CT',
    'DELAWARE': 'DE', 'DISTRICT OF COLUMBIA': 'DC', 'FLORIDA': 'FL',
    'GEORGIA': 'GA', 'HAWAII': 'HI', 'IDAHO': 'ID', 'ILLINOIS': 'IL',
    'INDIANA': 'IN', 'IOWA': 'IA', 'KANSAS': 'KS', 'KENTUCKY': 'KY',
    'LOUISIANA': 'LA', 'MAINE': 'ME', 'MARYLAND': 'MD',
    'MASSACHUSETTS': 'MA', 'MICHIGAN': 'MI', 'MINNESOTA': 'MN',
    'MISSISSIPPI': 'MS', 'MISSOURI': 'MO', 'MONTANA': 'MT',
    'NEBRASKA': 'NE', 'NEVADA': 'NV', 'NEW HAMPSHIRE': 'NH',
    'NEW JERSEY': 'NJ', 'NEW MEXICO': 'NM', 'NEW YORK': 'NY',
    'NORTH CAROLINA': 'NC', 'NORTH DAKOTA': 'ND', 'OHIO': 'OH',
    'OKLAHOMA': 'OK', 'OREGON': 'OR', 'PENNSYLVANIA': 'PA',
    'RHODE ISLAND': 'RI', 'SOUTH CAROLINA': 'SC',
    'SOUTH DAKOTA': 'SD', 'TENNESSEE': 'TN', 'TEXAS': 'TX',
    'UTAH': 'UT', 'VERMONT': 'VT', 'VIRGINIA': 'VA',
    'WASHINGTON': 'WA', 'WEST VIRGINIA': 'WV',
    'WISCONSIN': 'WI', 'WYOMING': 'WY'
}

# Create State column matching incident dataset
enrollment_df["State"] = enrollment_df["state"].map(state_abbrev)

# Rename year column for merge consistency
enrollment_df = enrollment_df.rename(columns={"year": "Year"})

In [13]:
# Keep only the larger enrollment value per state–year
enrollment_df = (
    enrollment_df
    .sort_values("total_students", ascending=False)
    .drop_duplicates(["State", "Year"])
)

In [14]:
# Merge incident counts with enrollment data on State and Year
merged_df_table = incident_counts.merge(
    enrollment_df[["State", "Year", "total_students"]],
    on=["State", "Year"],
    how="left"
)

# Check for any unmatched rows after merge
print(merged_df_table["total_students"].isna().sum())

# Compute incident rate per 100,000 students
merged_df_table["incident_rate_per_100k"] = (
    merged_df_table["incident_count"] / merged_df_table["total_students"]
) * 100000

# Inspect result
print(merged_df_table.head())

0
  State  Year  incident_count  total_students  incident_rate_per_100k
0    AK  1997               1          129919                0.769710
1    AK  2005               1          131701                0.759296
2    AK  2018               1          129660                0.771248
3    AK  2019               1          127641                0.783447
4    AK  2022               1          126457                0.790783


At state-year level, severity should not be modeled per incident.

In [15]:
incident_df_from_1987 = incident_df_from_1987[incident_df_from_1987["State"].str.match(r"^[A-Z]{2}$", na=False)]
print("Remaining rows:", len(incident_df_from_1987))

Remaining rows: 3070


In [16]:
valid_states = incident_df_from_1987["State"].str.match(r"^[A-Z]{2}$", na=False)
print("Valid rows:", valid_states.sum())
print("Invalid rows:", (~valid_states).sum())

Valid rows: 3070
Invalid rows: 0


In [17]:
print(incident_df_from_1987.head())

     Incident_ID  Month  Day  Year                       Date  \
0  19660311NCIRC      3   11  1966  1966-03-11T00:00:00+00:00   
1  19660314TXCAW      3   14  1966  1966-03-14T00:00:00+00:00   
2  19660324CACAM      3   24  1966  1966-03-24T00:00:00+00:00   
3  19660328CAJOL      3   28  1966  1966-03-28T00:00:00+00:00   
4  19660427NYBAB      4   27  1966  1966-04-27T00:00:00+00:00   

                             School  Victims_Killed Victims_Wounded  \
0  Irwing Avenue Junior High School               0               1   
1                Carver High School               1               0   
2    Camino Pablo Elementary School               3               0   
3                Jordan High School               0               1   
4             Bay Shore High School               1               0   

   Number_Victims Shooter_Killed  ... Preplanned SRO_School  \
0               1              0  ...         No              
1               1              0  ...         No        

In [18]:
print(merged_df_table["incident_count"].describe())

count    861.000000
mean       3.191638
std        3.745599
min        1.000000
25%        1.000000
50%        2.000000
75%        4.000000
max       25.000000
Name: incident_count, dtype: float64


In [19]:
print(merged_df_table["total_students"].describe())

count    8.610000e+02
mean     1.339581e+06
std      1.313322e+06
min      5.003200e+04
25%      5.573810e+05
50%      8.762130e+05
75%      1.662643e+06
max      6.322586e+06
Name: total_students, dtype: float64


In [20]:
import numpy as np

merged_df_table["log_enrollment"] = np.log(merged_df_table["total_students"])
print(merged_df_table["log_enrollment"].describe())

count    861.000000
mean      13.701213
std        0.947233
min       10.820418
25%       13.231004
50%       13.683364
75%       14.323919
max       15.659639
Name: log_enrollment, dtype: float64


In [21]:
incident_agg = (
    incident_df_from_1987
    .groupby(["State","Year"], as_index=False)
    .size()
    .rename(columns={"size": "incident_count"})
)

merged_df_table = enrollment_df.merge(
    incident_agg,
    on=["State","Year"],
    how="left"
)

merged_df_table["incident_count"] = merged_df_table["incident_count"].fillna(0)

print(merged_df_table.shape)
print((merged_df_table["incident_count"] == 0).sum())

(1989, 5)
1128


In [22]:
merged_df_table["log_enrollment"] = np.log(merged_df_table["total_students"])
print(merged_df_table.head())

        state  Year  total_students State  incident_count  log_enrollment
0  CALIFORNIA  2005         6322586    CA             6.0       15.659639
1  CALIFORNIA  2006         6312103    CA             4.0       15.657979
2  CALIFORNIA  2004         6298928    CA             7.0       15.655890
3  CALIFORNIA  2007         6274813    CA             5.0       15.652054
4  CALIFORNIA  2003         6244403    CA             4.0       15.647196


In [23]:
merged_df_table = merged_df_table.drop(columns=["state"])
print(merged_df_table.columns)

Index(['Year', 'total_students', 'State', 'incident_count', 'log_enrollment'], dtype='str')


In [24]:
all_rows = []
start = 0
batch_size = 1000

while True:
    response = (
        supabase
        .table("background_check_binary")
        .select("*")
        .range(start, start + batch_size - 1)
        .execute()
    )
    
    data = response.data
    
    if not data:
        break
        
    all_rows.extend(data)
    start += batch_size

bg_df = pd.DataFrame(all_rows)

print(bg_df.shape)

(2295, 3)


In [25]:
policy_tables = [
    "background_check_binary",
    "permit_to_purchase_binary",
    "prohibited_possessor_binary",
    "waiting_period_binary",
    "registration_binary",
    "child_access_binary",
    "safety_training_binary",
    "report_stolen_lost_binary",
    "firearm_sales_retrictions_binary",
    "k12_settings_binary"
]

policy_dfs = {}

for table in policy_tables:
    all_rows = []
    start = 0
    batch_size = 1000

    while True:
        response = (
            supabase
            .table(table)
            .select("*")
            .range(start, start + batch_size - 1)
            .execute()
        )
        data = response.data
        if not data:
            break
        all_rows.extend(data)
        start += batch_size

    policy_dfs[table] = pd.DataFrame(all_rows)

print({k: v.shape for k, v in policy_dfs.items()})

{'background_check_binary': (2295, 3), 'permit_to_purchase_binary': (2115, 3), 'prohibited_possessor_binary': (2115, 3), 'waiting_period_binary': (2295, 3), 'registration_binary': (2295, 3), 'child_access_binary': (1620, 3), 'safety_training_binary': (405, 3), 'report_stolen_lost_binary': (765, 3), 'firearm_sales_retrictions_binary': (2295, 3), 'k12_settings_binary': (315, 3)}


In [26]:
for name, df in policy_dfs.items():
    merged_df_table = merged_df_table.merge(
        df,
        left_on=["State","Year"],
        right_on=["state","year"],
        how="left"
    )
    merged_df_table = merged_df_table.drop(columns=["state","year"])

# Replace NaN with 0 for all policy indicators
policy_cols = [col for col in merged_df_table.columns if col.endswith("_law")]
merged_df_table[policy_cols] = merged_df_table[policy_cols].fillna(0)

print(merged_df_table.shape)
print(len(policy_cols), "policy variables merged")

(1989, 15)
10 policy variables merged


## State–Year Negative Binomial Policy Model

### Objective

Estimate the association between state firearm policies and the **risk of school shooting incidents**, measured at the state–year level.

Risk is defined as:

\[
\text{Incident Rate}_{s,t} =
\frac{\text{Incident Count}_{s,t}}{\text{Student Enrollment}_{s,t}}
\]

---

### Data Structure

The dataset is a full state–year panel including:

- All U.S. states (including DC)  
- All years in the study window (1987–2025)  
- Zero-incident years included  
- Total student enrollment (exposure variable)  
- Binary indicators for firearm policy adoption  

Each row represents one **state–year observation**.

---

### Model Choice

A **Negative Binomial regression** is used because:

- The dependent variable is count data  
- The variance exceeds the mean (overdispersion)  
- Zero-incident years are common  

Enrollment is incorporated as a log-offset to model rates rather than raw counts.

---

### Model Specification

Dependent variable:
- `incident_count`

Offset:
- `log_enrollment`

Policy indicators:
- `background_check_law`  
- `permit_to_purchase_law`  
- `prohibited_possessor_law`  
- `waiting_period_law`  
- `registration_law`  
- `child_access_law`  
- `safety_training_law`  
- `report_stolen_lost_law`  
- `firearm_sales_retrictions_law`  
- `k12_settings_law`  

Fixed effects:
- `C(State)` (state fixed effects)  
- `C(Year)` (year fixed effects)  

Model form:

\[
\log(E[Y_{s,t}]) =
\beta_0 +
\beta_1 \text{Policy}_{1,s,t} +
\dots +
\alpha_s +
\lambda_t +
\log(\text{Enrollment}_{s,t})
\]

Where:

- \(\alpha_s\) = state fixed effects  
- \(\lambda_t\) = year fixed effects  

The offset ensures coefficients reflect effects on **incident rates**, not raw counts.

---

### Role of Fixed Effects

**State fixed effects** control for time-invariant differences across states, including:

- Baseline risk levels  
- Persistent cultural or institutional factors  
- Long-standing reporting differences  

**Year fixed effects** absorb nationwide shocks affecting all states in a given year, including:

- Changes in reporting templates  
- Adjustments in school universe definitions  
- Federal classification revisions  
- National secular trends  

This structure mitigates instrumentation bias from national reporting changes by ensuring identification comes from **within-state changes relative to other states in the same year**, rather than from raw national trend shifts.

---

### Interpretation

Exponentiated coefficients are **Incidence Rate Ratios (IRR)**:

- IRR < 1 → associated with lower incident rate  
- IRR > 1 → associated with higher incident rate  

Estimates represent conditional associations controlling for:

- Enrollment exposure  
- Other firearm policies  
- State-specific fixed factors  
- National year-specific shocks  

---

### Methodological Caveats

Policy adoption may be endogenous:

- Laws may be passed in response to prior incidents  
- Positive coefficients may reflect reverse causality  

Additionally:

- Identification relies on within-state temporal variation  
- If reporting quality varies across states over time (not uniformly nationwide), residual measurement bias may remain  

Results should therefore be interpreted as **associative rather than causal**.

Potential extensions:

- Cluster-robust standard errors at the state level  
- Lagged policy variables  
- Event-study framework  
- State-specific linear trends  

In [27]:
formula = """
incident_count ~ 
background_check_law +
permit_to_purchase_law +
prohibited_possessor_law +
waiting_period_law +
registration_law +
child_access_law +
safety_training_law +
report_stolen_lost_law +
firearm_sales_retrictions_law +
k12_settings_law +
C(State) +
C(Year)
"""

model_joint = smf.glm(
    formula=formula,
    data=merged_df_table,
    family=sm.families.NegativeBinomial(),
    offset=merged_df_table["log_enrollment"]
).fit()

print(model_joint.summary())

/home/tommie-clark/capstoneProje t/capstoneProject2026/.venv/lib/python3.12/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


                 Generalized Linear Model Regression Results                  
Dep. Variable:         incident_count   No. Observations:                 1989
Model:                            GLM   Df Residuals:                     1890
Model Family:        NegativeBinomial   Df Model:                           98
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -2247.4
Date:                Fri, 13 Mar 2026   Deviance:                       1055.1
Time:                        11:31:12   Pearson chi2:                 1.36e+03
No. Iterations:                     9   Pseudo R-squ. (CS):             0.4500
Covariance Type:            nonrobust                                         
                                    coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------
Intercept     

In [28]:
import numpy as np
print(np.exp(model_joint.params))

Intercept                        2.811915e-07
C(State)[T.AL]                   2.009817e+00
C(State)[T.AR]                   1.349513e+00
C(State)[T.AZ]                   6.003576e-01
C(State)[T.CA]                   1.453006e+00
                                     ...     
child_access_law                 1.045549e+00
safety_training_law              8.908755e-01
report_stolen_lost_law           7.385070e-01
firearm_sales_retrictions_law    1.009689e+00
k12_settings_law                 6.729423e-01
Length: 99, dtype: float64


## Two-Way Fixed Effects Model (State + Year)

### Objective

Strengthen the policy model by controlling for both:

- **State fixed effects** (time-invariant state characteristics)
- **Year fixed effects** (national shocks and time trends)

This produces a **two-way fixed effects (TWFE)** panel specification.

---

### Why Add Year Fixed Effects?

Year effects control for:

- National increases in school shooting incidents
- Federal policy changes
- Media amplification cycles
- Reporting changes over time
- Macro-level social shifts

Without year controls, policy variables may absorb national time trends.

---

### Model Specification

The model now includes:

- `C(State)` → controls baseline state differences  
- `C(Year)` → controls nationwide year-specific shocks  
- Policy indicators (binary)
- Offset: `log(total_students)`

Formally:

\[
\log(E[Y_{s,t}]) =
\beta_0 +
\beta_k \text{Policy}_{k,s,t} +
\gamma_s +
\delta_t +
\log(\text{Enrollment}_{s,t})
\]

Where:

- \( \gamma_s \) = state fixed effects  
- \( \delta_t \) = year fixed effects  

---

### Interpretation

Policy coefficients now represent:

> Within-state changes in incident rates,
> relative to that state's baseline,
> after accounting for national time shocks.

This is a substantially more rigorous specification than earlier models.

---

### Key Result

After adding year fixed effects:

- Many previously significant policy effects attenuate
- Several coefficients lose statistical significance
- Model fit improves substantially (Pseudo R² increased to ~0.45)

This indicates that:

- National time trends explain a large portion of variation
- Some earlier policy effects were partly capturing time dynamics

---

### Methodological Implication

The TWFE model provides:

- Stronger control for confounding
- More credible within-state identification
- Reduced risk of attributing national trends to policy effects

However, policies may still be endogenous (adopted in response to prior incidents).

Next methodological extensions may include:

- Lagged policy indicators
- Event-study design
- Dynamic adoption modeling

In [29]:
formula = """
incident_count ~ 
background_check_law +
permit_to_purchase_law +
prohibited_possessor_law +
waiting_period_law +
registration_law +
child_access_law +
safety_training_law +
report_stolen_lost_law +
firearm_sales_retrictions_law +
k12_settings_law +
C(State) +
C(Year)
"""

model_twfe = smf.glm(
    formula=formula,
    data=merged_df_table,
    family=sm.families.NegativeBinomial(),
    offset=merged_df_table["log_enrollment"]
).fit()

print(model_twfe.summary())


/home/tommie-clark/capstoneProje t/capstoneProject2026/.venv/lib/python3.12/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


                 Generalized Linear Model Regression Results                  
Dep. Variable:         incident_count   No. Observations:                 1989
Model:                            GLM   Df Residuals:                     1890
Model Family:        NegativeBinomial   Df Model:                           98
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -2247.4
Date:                Fri, 13 Mar 2026   Deviance:                       1055.1
Time:                        11:31:12   Pearson chi2:                 1.36e+03
No. Iterations:                     9   Pseudo R-squ. (CS):             0.4500
Covariance Type:            nonrobust                                         
                                    coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------
Intercept     

## Cluster-Robust Inference (State-Level)


### Why Cluster Standard Errors?

This is a state × year panel, so errors can be serially correlated within states across years.
Clustering by state accounts for within-state dependence in the residual process.
This changes inference (standard errors and p-values), not coefficient point estimates.
State-clustered standard errors are typically more conservative and methodologically appropriate for this panel structure.


In [30]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm

# ================================
# STEP 1 — Fit Poisson TWFE Model
# ================================

model_twfe = smf.glm(
    formula=formula,
    data=merged_df_table,
    family=sm.families.Poisson(),
    offset=merged_df_table["log_enrollment"]
).fit()

# ================================
# STEP 2 — Fit Clustered Version
# ================================

model_twfe_cluster = smf.glm(
    formula=formula,
    data=merged_df_table,
    family=sm.families.Poisson(),
    offset=merged_df_table["log_enrollment"]
).fit(
    cov_type="cluster",
    cov_kwds={"groups": merged_df_table["State"]}
)

print("Two-Way FE Poisson with State-Clustered SE")
print(model_twfe_cluster.summary())

# ================================
# STEP 3 — Policy-Only Comparison
# ================================

policy_variables = [v for v in model_twfe.params.index if v.endswith("_law")]

se_comparison = pd.DataFrame({
    "variable": policy_variables,
    "coef": [model_twfe.params[v] for v in policy_variables],
    "se_nonclustered": [model_twfe.bse[v] for v in policy_variables],
    "se_clustered": [model_twfe_cluster.bse[v] for v in policy_variables],
    "p_nonclustered": [model_twfe.pvalues[v] for v in policy_variables],
    "p_clustered": [model_twfe_cluster.pvalues[v] for v in policy_variables],
})

print("\nStandard Error Comparison")
print(se_comparison)



Two-Way FE Poisson with State-Clustered SE
                 Generalized Linear Model Regression Results                  
Dep. Variable:         incident_count   No. Observations:                 1989
Model:                            GLM   Df Residuals:                     1890
Model Family:                 Poisson   Df Model:                           98
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -2072.7
Date:                Fri, 13 Mar 2026   Deviance:                       1831.5
Time:                        11:31:13   Pearson chi2:                 2.22e+03
No. Iterations:                     8   Pseudo R-squ. (CS):             0.8208
Covariance Type:              cluster                                         
                                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------

## Lag the policy variables.

We test whether policy adoption predicts future incident risk rather than same-year effects.

In [31]:
policy_cols = [
    "background_check_law",
    "permit_to_purchase_law",
    "prohibited_possessor_law",
    "waiting_period_law",
    "registration_law",
    "child_access_law",
    "safety_training_law",
    "report_stolen_lost_law",
    "firearm_sales_retrictions_law",
    "k12_settings_law"
]

# sort properly first
merged_df_table = merged_df_table.sort_values(["State", "Year"])

# create 1-year lags within state
for col in policy_cols:
    merged_df_table[col + "_lag1"] = (
        merged_df_table.groupby("State")[col].shift(1)
    )

print(
    merged_df_table[[c for c in merged_df_table.columns if "_lag1" in c]].isna().sum().head()
)

background_check_law_lag1        51
permit_to_purchase_law_lag1      51
prohibited_possessor_law_lag1    51
waiting_period_law_lag1          51
registration_law_lag1            51
dtype: int64


In [32]:
lag_cols = [col for col in merged_df_table.columns if col.endswith("_lag1")]

merged_df_table[lag_cols] = merged_df_table[lag_cols].fillna(0)

print(merged_df_table[lag_cols].isna().sum().sum())

0


Notice the Year coefficients:

2018–2024 effects are extremely large and significant.

This indicates:

National upward structural shift in incidents dominates policy variation.

That is your strongest empirical finding so far.

evidence currently supports:

Strong national time dynamics

Weak within-state short-run policy effects

Significant endogeneity concerns

That is a legitimate, publishable conclusion if framed correctly.

In [33]:
formula_lag = """
incident_count ~ 
background_check_law_lag1 +
permit_to_purchase_law_lag1 +
prohibited_possessor_law_lag1 +
waiting_period_law_lag1 +
registration_law_lag1 +
child_access_law_lag1 +
safety_training_law_lag1 +
report_stolen_lost_law_lag1 +
firearm_sales_retrictions_law_lag1 +
k12_settings_law_lag1 +
C(State) +
C(Year)
"""

model_twfe_lag = smf.glm(
    formula=formula_lag,
    data=merged_df_table,
    family=sm.families.NegativeBinomial(),
    offset=merged_df_table["log_enrollment"]
).fit()

print(model_twfe_lag.summary())

/home/tommie-clark/capstoneProje t/capstoneProject2026/.venv/lib/python3.12/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


                 Generalized Linear Model Regression Results                  
Dep. Variable:         incident_count   No. Observations:                 1989
Model:                            GLM   Df Residuals:                     1890
Model Family:        NegativeBinomial   Df Model:                           98
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -2251.4
Date:                Fri, 13 Mar 2026   Deviance:                       1063.2
Time:                        11:31:13   Pearson chi2:                 1.34e+03
No. Iterations:                     9   Pseudo R-squ. (CS):             0.4478
Covariance Type:            nonrobust                                         
                                         coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
Inte

## Switching from Incident Frequency to Severity

### Motivation

Prior models examined whether firearm policies were associated with the **frequency of school shooting incidents**.

However, policies may not primarily affect how often incidents occur.  
Instead, they may influence the **severity of incidents when they do occur**.

Severity is operationalized as:

\[
\text{Total Victims}_{s,t}
\]

Where total victims includes both fatalities and injuries aggregated at the state–year level.

---

### Why Model Severity?

There are theoretical reasons to expect differential effects:

- Some policies may reduce lethality rather than prevent incidents
- Emergency response policies may affect victim counts
- Restrictions may alter weapon access but not eliminate events
- Frequency and severity are distinct processes

Frequency answers:
> Do policies reduce the number of incidents?

Severity answers:
> Conditional on incidents occurring, do policies reduce harm?

---

### Statistical Approach

Severity is also count data and exhibits:

- Overdispersion
- Heavy right tail (extreme events)
- Many low-severity years

Therefore, we continue using a:

- **Negative Binomial model**
- With state and year fixed effects
- With enrollment offset when modeling rates per student

---

### Conceptual Difference

Frequency model:
\[
Y_{s,t} = \text{Incident Count}
\]

Severity model:
\[
Y_{s,t} = \text{Total Victims}
\]

The interpretation shifts from:
> Policy impact on event probability

to:
> Policy impact on harm intensity

---

This allows a more nuanced assessment of firearm policy effects.

In [34]:
print(merged_df_table.columns)

Index(['Year', 'total_students', 'State', 'incident_count', 'log_enrollment',
       'background_check_law', 'permit_to_purchase_law',
       'prohibited_possessor_law', 'waiting_period_law', 'registration_law',
       'child_access_law', 'safety_training_law', 'report_stolen_lost_law',
       'firearm_sales_retrictions_law', 'k12_settings_law',
       'background_check_law_lag1', 'permit_to_purchase_law_lag1',
       'prohibited_possessor_law_lag1', 'waiting_period_law_lag1',
       'registration_law_lag1', 'child_access_law_lag1',
       'safety_training_law_lag1', 'report_stolen_lost_law_lag1',
       'firearm_sales_retrictions_law_lag1', 'k12_settings_law_lag1'],
      dtype='str')


In [35]:
severity_df = (
    incident_df_from_1987
    .groupby(["State", "Year"])["Number_Victims"]
    .sum()
    .reset_index()
    .rename(columns={"Number_Victims": "total_victims"})
)

merged_df_table = merged_df_table.merge(
    severity_df,
    on=["State", "Year"],
    how="left"
)

merged_df_table["total_victims"] = merged_df_table["total_victims"].fillna(0)

print(merged_df_table[["incident_count", "total_victims"]].head())

   incident_count  total_victims
0             0.0            0.0
1             0.0            0.0
2             0.0            0.0
3             0.0            0.0
4             0.0            0.0


## Two-Way Fixed Effects Model – Severity (Total Victims)

### Objective

The dependent variable was shifted from **incident frequency** to **severity**, defined as:

> **Total victims per state-year**, scaled by student enrollment.

This allows evaluation of whether firearm policies are associated not only with the number of incidents, but with the *magnitude of harm* conditional on exposure.

---

### Model Specification

A Negative Binomial GLM was estimated:

total_victims ~  
background_check_law +  
permit_to_purchase_law +  
prohibited_possessor_law +  
waiting_period_law +  
registration_law +  
child_access_law +  
safety_training_law +  
report_stolen_lost_law +  
firearm_sales_restrictions_law +  
k12_settings_law +  
C(State) +  
C(Year)

- **Outcome**: `total_victims`
- **Offset**: `log_enrollment`
- **State Fixed Effects**: Control for time-invariant state characteristics
- **Year Fixed Effects**: Control for national time shocks and common trends
- **Model Family**: Negative Binomial (accounts for overdispersion in count data)

---

### Model Fit

- Observations: 1,989  
- Log-Likelihood: -2292.4  
- Pseudo R² (Cox–Snell): 0.381  

The inclusion of state and year fixed effects substantially improves explanatory power relative to simpler specifications.

---

### Policy Effects

After controlling for state and year fixed effects:

- No policy variable is statistically significant at p < 0.05.
- Marginal effects:
  - `waiting_period_law` (p ≈ 0.066)
  - `registration_law` (p ≈ 0.060)

Once fixed effects are included, previously significant policy coefficients attenuate.

---

### Interpretation

These results suggest:

- Much of the variation in severity is driven by:
  - Persistent structural differences across states
  - Strong national temporal effects (particularly post-2018)
- Binary policy adoption indicators do not exhibit strong within-state effects in this specification.

This indicates that cross-state differences and national trends dominate the signal relative to contemporaneous policy changes.

---

### Data Note

`C(State)[T.WY]` exhibits instability due to sparse victim counts, suggesting potential quasi-separation in low-incidence states. This may warrant:

- Dropping extremely low-count states  
- Aggregating smaller states  
- Considering zero-inflated or alternative count models  

---

### Summary

The two-way fixed effects severity model indicates:

- Strong year effects, especially in recent years
- Limited robust evidence that binary policy adoption alone reduces total victim counts once structural controls are applied
- Severity appears more influenced by structural and temporal factors than by contemporaneous statutory adoption alone

In [36]:
formula_severity = """
total_victims ~ 
background_check_law +
permit_to_purchase_law +
prohibited_possessor_law +
waiting_period_law +
registration_law +
child_access_law +
safety_training_law +
report_stolen_lost_law +
firearm_sales_retrictions_law +
k12_settings_law +
C(State) +
C(Year)
"""

model_severity_twfe = smf.glm(
    formula=formula_severity,
    data=merged_df_table,
    family=sm.families.NegativeBinomial(),
    offset=merged_df_table["log_enrollment"]
).fit()

print(model_severity_twfe.summary())

/home/tommie-clark/capstoneProje t/capstoneProject2026/.venv/lib/python3.12/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


                 Generalized Linear Model Regression Results                  
Dep. Variable:          total_victims   No. Observations:                 1989
Model:                            GLM   Df Residuals:                     1890
Model Family:        NegativeBinomial   Df Model:                           98
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -2292.4
Date:                Fri, 13 Mar 2026   Deviance:                       1638.1
Time:                        11:31:14   Pearson chi2:                 3.07e+03
No. Iterations:                    25   Pseudo R-squ. (CS):             0.3810
Covariance Type:            nonrobust                                         
                                    coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------
Intercept     

In [37]:
severity_df = merged_df_table.copy()
severity_df = severity_df[severity_df["incident_count"] > 0].copy()

severity_df["victims_per_incident"] = (
    severity_df["total_victims"] / severity_df["incident_count"]
)

In [38]:
severity_df = severity_df[severity_df["victims_per_incident"] > 0].copy()

## Incident-Level Severity Model (Victims per Incident)

### Objective

Estimate whether firearm policies affect **how severe an incident is**, conditional on at least one incident occurring.

Severity is defined as:

victims_per_incident = total_victims / incident_count

The sample is restricted to state-years with incident_count > 0 and victims_per_incident > 0.

---

### Model Specification

A Gamma GLM with log link was estimated:

victims_per_incident ~
background_check_law +
permit_to_purchase_law +
prohibited_possessor_law +
waiting_period_law +
registration_law +
child_access_law +
safety_training_law +
report_stolen_lost_law +
firearm_sales_retrictions_law +
k12_settings_law +
C(State) +
C(Year)

- Family: Gamma
- Link: Log
- State fixed effects included
- Year fixed effects included
- Observations: 691
- Pseudo R² (Cox–Snell): 0.384

The Gamma model is appropriate because:
- The outcome is strictly positive
- The distribution is right-skewed
- Variance increases with the mean

---

### Key Findings

After controlling for state and year fixed effects:

- `k12_settings_law` is positive and statistically significant (p < 0.01)
- `background_check_law` approaches marginal significance (p ≈ 0.10)
- All other policy variables are not statistically significant

Year effects are strongly negative in the post-2000 period relative to the omitted base year, suggesting changes in incident composition over time.

---

### Interpretation

Conditional on an incident occurring:

- Most firearm policy indicators do not show strong evidence of reducing victims per incident.
- K–12 settings restrictions are associated with higher victims per incident in this specification.
- Temporal effects appear more structurally important than contemporaneous policy adoption.

This suggests that:
- Policies may influence incident frequency more than conditional lethality,
- Or severity dynamics are driven by factors not captured by binary statutory indicators.

---

### Position in Overall Framework

You have now estimated:

1. Frequency model (incidents per enrollment)
2. Severity model (total victims per enrollment)
3. Conditional severity model (victims per incident)

This completes a three-layer decomposition:

Total Harm =
Incident Frequency × Severity per Incident

In [39]:
formula_incident_severity = """
victims_per_incident ~
background_check_law +
permit_to_purchase_law +
prohibited_possessor_law +
waiting_period_law +
registration_law +
child_access_law +
safety_training_law +
report_stolen_lost_law +
firearm_sales_retrictions_law +
k12_settings_law +
C(State) +
C(Year)
"""

model_incident_severity = smf.glm(
    formula=formula_incident_severity,
    data=severity_df,
    family=sm.families.Gamma(link=sm.families.links.log())
).fit()

print(model_incident_severity.summary())

/home/tommie-clark/capstoneProje t/capstoneProject2026/.venv/lib/python3.12/site-packages/statsmodels/genmod/families/links.py:13: FutureWarning: The log link alias is deprecated. Use Log instead. The log link alias will be removed after the 0.15.0 release.
  warnings.warn(


                  Generalized Linear Model Regression Results                   
Dep. Variable:     victims_per_incident   No. Observations:                  691
Model:                              GLM   Df Residuals:                      593
Model Family:                     Gamma   Df Model:                           97
Link Function:                      log   Scale:                         0.54449
Method:                            IRLS   Log-Likelihood:                -710.65
Date:                  Fri, 13 Mar 2026   Deviance:                       259.29
Time:                          11:31:15   Pearson chi2:                     323.
No. Iterations:                     100   Pseudo R-squ. (CS):             0.3838
Covariance Type:              nonrobust                                         
                                    coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------